To train a neural network, use this order:


    Prepare the dataset.

    Split data into train, validation, and test sets.

    Build the neural network architecture.

    Initialize weights and biases.

    Choose a loss function.

    Choose an optimizer.

    Forward pass.

    Backward pass.

    Gradient descent.

    Epochs and batches.

    Learning rate.

    Evaluate on validation data.

    Overfitting and regularization.

    Early stopping.

    Save and load the model.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import torch

In [2]:
df=pd.read_csv("telecom_churn_dataset.csv")
df.head()

,Tenure_Months,Contract_Type,Internet_Service,Monthly_Charges,Total_Charges,Customer_Service_Calls,Churn
0,52,2,1,104.90,5434.92,2,0
1,15,2,1,88.51,1289.27,3,0
2,61,0,0,61.26,3775.28,2,0
3,21,1,1,79.97,1735.82,1,0
4,24,0,1,82.01,1922.62,3,0


In [3]:
df.isna().sum()

Tenure_Months             0
Contract_Type             0
Internet_Service          0
Monthly_Charges           0
Total_Charges             0
Customer_Service_Calls    0
Churn                     0
dtype: int64

In [4]:
df.describe()

,Tenure_Months,Contract_Type,Internet_Service,Monthly_Charges,Total_Charges,Customer_Service_Calls,Churn
count,1000.000000,1000.00000,1000.000000,1000.000000,1000.000000,1000.00000,1000.000000
mean,35.489000,0.91200,0.700000,73.178090,2612.653460,1.77300,0.211000
std,20.709485,0.82882,0.458487,20.788936,1727.478376,1.10937,0.408223
min,1.000000,0.00000,0.000000,20.000000,20.000000,0.00000,0.000000
25%,17.000000,0.00000,0.000000,55.452500,1138.695000,1.00000,0.000000
50%,35.000000,1.00000,1.000000,79.395000,2454.900000,2.00000,0.000000
75%,54.000000,2.00000,1.000000,88.390000,3869.047500,2.00000,0.000000
max,71.000000,2.00000,1.000000,112.660000,7460.210000,6.00000,1.000000


In [5]:
# Set seed for exact reproducibility
np.random.seed(42)
num_rows = 1000

# Generating features strictly between 0 and 10 to eliminate scaling requirements
tenure_years = np.random.uniform(0.0, 6.0, size=num_rows)
contract_score = np.random.choice([0.0, 1.0, 2.0], size=num_rows, p=[0.4, 0.3, 0.3])
internet_tier = np.random.choice([0.0, 1.0, 2.0], size=num_rows, p=[0.3, 0.5, 0.2])
bill_intensity = np.clip(np.random.normal(5.5, 1.5, size=num_rows), 1.0, 10.0)
support_tickets = np.random.binomial(n=5, p=0.3, size=num_rows).astype(float)

# Calculate a logical Churn target based on the features
logit = (0.6 * bill_intensity - 0.5 * tenure_years - 1.2 * contract_score + 0.8 * support_tickets + 0.3 * internet_tier - 1.5)
churn_probability = 1 / (1 + np.exp(-logit))
churn = np.where(np.random.rand(num_rows) < churn_probability, 1, 0)

# Build the finalized dataframe
df = pd.DataFrame({
    'Tenure_Years': np.round(tenure_years, 2),
    'Contract_Score': contract_score,
    'Internet_Tier': internet_tier,
    'Bill_Intensity': np.round(bill_intensity, 2),
    'Support_Tickets': support_tickets,
    'Churn': churn
})

# Save to your local folder
df.to_csv("clean_ready_churn_data.csv", index=False)
print("SUCCESS: 'clean_ready_churn_data.csv' has been created in your folder!")

SUCCESS: 'clean_ready_churn_data.csv' has been created in your folder!


In [6]:
df

,Tenure_Years,Contract_Score,Internet_Tier,Bill_Intensity,Support_Tickets,Churn
0,2.25,0.0,0.0,7.09,1.0,1
1,5.70,1.0,0.0,6.43,0.0,0
2,4.39,2.0,2.0,6.53,2.0,1
3,3.59,2.0,0.0,3.45,4.0,0
4,0.94,2.0,0.0,7.32,1.0,1
...,...,...,...,...,...,...
995,0.55,1.0,2.0,7.94,1.0,1
996,5.50,2.0,0.0,5.07,1.0,0
997,0.82,0.0,1.0,7.89,3.0,1
998,5.70,0.0,0.0,6.52,2.0,1


In [7]:
df.isna().sum()

X_tensors=df.drop("Churn",axis=1)
y_tensors=df["Churn"]



In [8]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X_tensors,y_tensors,test_size=0.2,random_state=42)
print("x_train = ",X_train.shape[0])
print("x_test = ",X_test.shape[0])


x_train =  800
x_test =  200


In [9]:
print("y_train = ",y_train.shape[0])
print("y_test = ",y_test.shape[0])

X_train = torch.tensor(X_train.values, dtype=torch.float32)

y_train = torch.tensor(y_train.values, dtype=torch.float32)
y_train = y_train.view(-1,1)



y_train =  800
y_test =  200


In [10]:
import torch.nn as nn
class ChrunClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.network=nn.Sequential(
            nn.Linear(5,16),
            nn.ReLU(),
            nn.Linear(16,8),
            nn.ReLU(),
            nn.Linear(8,1)
        )

    def forward(self, x):
        return self.network(x)
model = ChrunClassifier()


In [11]:
model

ChrunClassifier(
  (network): Sequential(
    (0): Linear(in_features=5, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=8, bias=True)
    (3): ReLU()
    (4): Linear(in_features=8, out_features=1, bias=True)
  )
)

In [12]:
import torch.optim as optim

# Loss: Binary Cross Entropy with Logits (expects raw numbers from the last layer)
criterion = nn.BCEWithLogitsLoss()

# Optimizer: Adam is a highly reliable choice for general optimization
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [13]:
epoch=100
for epoch in range(epoch):
    model.train()

    prediction = model(X_train)

    loss=criterion(prediction,y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epoch}] ──── Training Loss: {loss.item():.4f}")

Epoch [10/9] ──── Training Loss: 0.6335
Epoch [20/19] ──── Training Loss: 0.5388
Epoch [30/29] ──── Training Loss: 0.4712
Epoch [40/39] ──── Training Loss: 0.4591
Epoch [50/49] ──── Training Loss: 0.4515
Epoch [60/59] ──── Training Loss: 0.4486
Epoch [70/69] ──── Training Loss: 0.4477
Epoch [80/79] ──── Training Loss: 0.4470
Epoch [90/89] ──── Training Loss: 0.4463
Epoch [100/99] ──── Training Loss: 0.4453


In [14]:
X_test=torch.tensor(X_test.values,dtype=torch.float32)
y_test=torch.tensor(y_test.values,dtype=torch.float32)
y_test=y_test.view(-1,1)
model.eval() # Set model to evaluation mode
with torch.no_grad(): # Turn off Autograd to save memory
    # Get raw test predictions
    test_logits = model(X_test)

    # Convert raw outputs to distinct probabilities (0 to 1) using Sigmoid
    test_probs = torch.sigmoid(test_logits)

    # Convert probabilities to binary predictions (True/False or 1/0)
    final_predictions = (test_probs >= 0.5).float()

    # Calculate total accuracy
    accuracy = (final_predictions == y_test).float().mean()
    print(f"\nFinal Test Set Accuracy: {accuracy.item() * 100:.2f}%")


Final Test Set Accuracy: 77.50%
